# UFC Fighter Rating

This notebook runs the whole pipeline and shows its two products:

1. **Fight-outcome models**: how well can the result of a UFC fight be predicted from what was known before it?
2. **Division rankings**: the active fighters of each division ranked by the model, and a test of that ranking against the official UFC rankings on real fights.

Each step writes its output to `data/processed/`. The three analysis notebooks read those files, so run this one first:

| Notebook | Content |
|---|---|
| [01_exploration](notebooks/01_exploration.ipynb) | The data: coverage, the corner artefact, how fights end, rounds, judges, the betting market, official rankings, professional records |
| [02_feature_engineering](notebooks/02_feature_engineering.ipynb) | How every feature is built without leaking the future, how much signal each one carries, and whether the extra data help |
| [03_models_and_rankings](notebooks/03_models_and_rankings.ipynb) | Model diagnostics, the ranking backtest in detail, Elo tuning |

The same pipeline runs from the command line with `python -m ufc_rating.pipeline`.

## 0. Configuration

In [1]:
# Data
REFRESH_DATA = False      # True: refresh Kaggle, Wikipedia and bestfightodds first (network; Kaggle API credentials)
SCRAPE_UFCSTATS = False   # True: also scrape ufcstats.com for events newer than the data (needs the 'scrape' extra)

# Who gets ranked
ACTIVE_DAYS = 730   # fought in the last two years
MIN_FIGHTS = 5      # at least five UFC fights
TOP_N = 10          # fighters shown per division

In [2]:
import pandas as pd
from IPython.display import Markdown, display

from ufc_rating.config import DIVISIONS
from ufc_rating.models.training import format_scores
from ufc_rating.pipeline import (
    backtest_rankings, build_features, compare_feature_groups, fit_models, rank_fighters, update_data,
)
from ufc_rating.ranking.backtest import format_summary

pd.set_option('display.width', 140)

## 1. Data

| Source | Content |
|---|---|
| Kaggle mirror of [ufcstats.com](http://ufcstats.com) (CC0) | Results, fight totals, round-by-round statistics, judges' scores, bonuses, fighter profiles. Only UFC events are kept. |
| Our ufcstats.com scraper | The events more recent than the mirror, in the same format (`data/raw/scraped/`) |
| Ultimate UFC Dataset (Kaggle, CC BY 4.0) | Betting odds and official ranks at fight time, 2010 to March 2026 |
| [bestfightodds.com](https://www.bestfightodds.com) | Closing odds the Kaggle source misses since 2023, and of every later event (median over the sportsbooks) |
| English Wikipedia (CC BY-SA 4.0) | The official rankings week by week since 2018, and the professional records of the fighters |

The raw files are versioned in `data/raw/` as they stood on 19 September 2026, so the results below can be reproduced exactly. Odds and rankings are matched to the fights on fighter names and dates; details in [01_exploration](notebooks/01_exploration.ipynb).

In [3]:
master, rounds = update_data(refresh=REFRESH_DATA, scrape=SCRAPE_UFCSTATS)

Master table: 8,905 UFC fights, 1993-11-12 to 2026-09-19; 20,904 rounds with stats


## 2. Features

Every decided fight becomes a comparison between two fighters **A** and **B**, drawn at random from the two corners, described by the difference of their career statistics, physical attributes and Elo rating **before** the fight. Draws, no contests and fights involving a UFC debutant (no history to learn from) are left out.

The Elo rating is updated after every fight (K = 80, split and majority decisions counting half). Why the corners have to be randomised, how each feature avoids leaking the future, and the round, judges and bonus features that were tested but not kept: [02_feature_engineering](notebooks/02_feature_engineering.ipynb).

In [4]:
elo_history, matchups, profiles = build_features(master, rounds)
ablation = compare_feature_groups(matchups)   # written to data/processed/, shown in notebook 02

Matchups: 6,525 fights with two experienced fighters, 24 stat features (+ odds)


Feature groups (rolling-origin log loss before the test period):
                             mean log loss  vs first set  folds better
features                                                              
model features                      0.6584        0.0000             0
+ rounds                            0.6583       -0.0002             2
+ judges                            0.6584       -0.0000             3
+ bonuses                           0.6579       -0.0005             4
+ rounds + judges + bonuses         0.6577       -0.0007             2


## 3. Models

The fights are split in time: the oldest 70% for training, the next 15% for validation, the most recent 15% for the final test. Four models (logistic regression, SVM, random forest, XGBoost) are tuned by time-ordered cross-validation on the training period, in two versions:

- **stats**: fighter data only. This is the model used for the rankings.
- **stats + odds**: the same inputs plus the bookmakers' implied probability.

The validation period picks the stats model used for the rankings. Every model is then refitted on training + validation, and the test period is scored once, at the end. For the rankings, which describe the fighters of today, the selected model is refitted on every fight.

In [5]:
fitted = fit_models(matchups)
results = fitted['results']

pd.DataFrame(results['split'], index=['from', 'to', 'fights']).T

Stats-only models:


  LogReg        CV log-loss 0.6607  {'C': 0.01}


  SVM           CV log-loss 0.6608  {'C': 0.01, 'kernel': 'linear'}


  RandomForest  CV log-loss 0.6681  {'max_depth': 8, 'min_samples_leaf': 10}


  XGBoost       CV log-loss 0.6691  {'learning_rate': 0.02, 'max_depth': 2}
Stats + odds models:
  LogReg        CV log-loss 0.6328  {'C': 0.01}


  SVM           CV log-loss 0.6365  {'C': 0.01, 'kernel': 'linear'}


  RandomForest  CV log-loss 0.6474  {'max_depth': 8, 'min_samples_leaf': 10}


  XGBoost       CV log-loss 0.6430  {'learning_rate': 0.02, 'max_depth': 2}


,from,to,fights
train,1994-03-11 00:00:00,2022-01-22 00:00:00,4565
validation,2022-02-05 00:00:00,2024-06-01 00:00:00,978
test,2024-06-08 00:00:00,2026-09-19 00:00:00,982


**Test period, all fights.** Accuracy is the share of winners correctly predicted (50% is a coin flip, since A is drawn at random). AUC measures how well the predicted probabilities rank winners above losers (0.5 = chance). Log loss and Brier score measure the quality of the probabilities themselves (lower is better). *Elo only* predicts with the pre-fight Elo ratings alone.

In [6]:
format_scores(results['test_stats'])

,Fights,Accuracy,AUC,Log loss,Brier
LogReg,982,66.4%,0.717,0.624,0.217
SVM,982,66.2%,0.717,0.623,0.217
RandomForest,982,65.8%,0.711,0.636,0.222
XGBoost,982,65.3%,0.709,0.632,0.221
Elo only,982,56.1%,0.591,0.681,0.244


**Against the betting market.** Only the test fights with odds, so every line is scored on exactly the same fights.

In [7]:
format_scores(results['test_market'])

,Fights,Accuracy,AUC,Log loss,Brier
Betting favourite (market),911,69.9%,0.758,0.586,0.200
Elo only,911,56.5%,0.599,0.678,0.243
LogReg (stats),911,66.4%,0.717,0.624,0.217
SVM (stats),911,66.3%,0.717,0.624,0.217
RandomForest (stats),911,66.1%,0.711,0.635,0.222
XGBoost (stats),911,65.4%,0.711,0.631,0.220
LogReg (stats + odds),911,69.8%,0.759,0.586,0.200
SVM (stats + odds),911,69.6%,0.759,0.586,0.200
RandomForest (stats + odds),911,70.0%,0.756,0.596,0.204
XGBoost (stats + odds),911,69.3%,0.759,0.587,0.200


Reading these two tables:

- Fighter statistics alone predict the winner two times out of three, clearly better than chance and better than Elo alone.
- The betting market stays the reference: bookmakers see everything the statistics see, plus injuries, camps, weight cuts and style match-ups.
- Adding the statistics to the odds brings the models level with the market, not above it: the public statistics carry no information the market has not already priced in.

Calibration, confidence and what each model relies on: [03_models_and_rankings](notebooks/03_models_and_rankings.ipynb).

## 4. Rankings

Active fighters (a fight in the last `ACTIVE_DAYS` days, at least `MIN_FIGHTS` UFC fights) are ranked in their most recent division by the **model**. In a virtual round-robin tournament, the selected stats model, refitted on all fights, predicts every possible match-up in the division; a fighter's score (*Model*) is their average win probability against the rest of the division.

Their **Elo** rating is shown alongside. Updated after every fight since 1993, it measures the record: *who* a fighter beat. The model already uses it as one of its inputs, together with age, physical attributes and career statistics.

In [8]:
as_of = master['date'].max()
best = results['best_stats_model']
rankings = rank_fighters(profiles, fitted['ranking_model'], as_of,
                         active_days=ACTIVE_DAYS, min_fights=MIN_FIGHTS)
print(f'Rankings as of {as_of.date()}, round-robin model: {best}')

Rankings as of 2026-09-19, round-robin model: LogReg


In [9]:
columns = ['Fighter', 'UFC record', 'Last fight', 'Model', 'Elo', 'Elo rank']
for division in DIVISIONS:
    table = rankings[rankings['division'] == division]
    if table.empty:
        continue
    display(Markdown(f'### {division} ({len(table)} active fighters)'))
    display(table.set_index('rank')[columns].head(TOP_N))

### Flyweight (35 active fighters)

,Fighter,UFC record,Last fight,Model,Elo,Elo rank
rank,,,,,,
1,Joshua Van,11-1,2026-09-19,0.805,1836,1
2,Tatsuro Taira,8-2,2026-05-09,0.731,1712,4
3,Manel Kape,8-3,2026-06-20,0.717,1745,2
4,Asu Almabayev,7-1,2026-06-27,0.662,1697,6
5,Sumudaerji,7-4,2026-08-29,0.652,1603,11
6,Andre Lima,5-1,2026-08-29,0.636,1591,14
7,Brandon Moreno,12-7-2,2026-09-12,0.626,1674,7
8,Alexandre Pantoja,14-5,2026-09-19,0.600,1720,3
9,Kyoji Horiguchi,9-2,2026-06-20,0.599,1706,5


### Bantamweight (53 active fighters)

,Fighter,UFC record,Last fight,Model,Elo,Elo rank
rank,,,,,,
1,Merab Dvalishvili,14-3,2025-12-06,0.781,1899,1
2,Umar Nurmagomedov,8-2,2026-08-29,0.776,1755,6
3,Raul Rosas Jr.,6-1,2026-03-07,0.767,1659,17
4,Sean O'Malley,12-3,2026-06-14,0.756,1824,3
5,Petr Yan,12-4,2025-12-06,0.732,1871,2
6,Montel Jackson,9-4,2026-04-25,0.717,1699,12
7,Farid Basharat,7-0,2026-07-11,0.704,1727,7
8,Song Yadong,13-4-1,2026-08-29,0.681,1799,4
9,Mario Bautista,12-3,2026-07-11,0.674,1789,5


### Featherweight (56 active fighters)

...

,Fighter,UFC record,Last fight,Model,Elo,Elo rank
rank,,,,,,
1,Tatiana Suarez,9-1,2026-04-11,0.802,1772,1
2,Fatima Kline,4-1,2026-07-18,0.796,1636,6
3,Iasmin Lucindo,5-2,2025-08-09,0.743,1614,9
4,Denise Gomes,7-2,2026-08-29,0.736,1689,4
5,Jaqueline Amorim,5-2,2026-05-30,0.663,1605,10
6,Gillian Robertson,14-7,2026-08-15,0.646,1672,5
7,Virna Jandiroba,9-4,2026-04-04,0.638,1724,3
8,Loopy Godinez,9-6,2026-04-11,0.607,1593,13
9,Mackenzie Dern,12-5,2026-08-15,0.602,1741,2


### Women's Flyweight (30 active fighters)

,Fighter,UFC record,Last fight,Model,Elo,Elo rank
rank,,,,,,
1,Natalia Silva,8-0,2026-01-24,0.771,1791,2
2,Erin Blanchfield,8-1,2025-11-15,0.746,1779,4
3,Valentina Shevchenko,15-3-1,2025-11-15,0.730,1877,1
4,Carli Judice,4-1,2026-08-22,0.701,1619,13
5,Maycee Barber,10-3,2026-03-28,0.665,1695,9
6,Casey O'Neill,7-2,2026-09-19,0.640,1644,11
7,Miranda Maverick,8-4,2025-06-14,0.633,1633,12
8,Zhang Weili,10-3,2025-11-15,0.632,1785,3
9,Manon Fiorot,8-2,2026-09-12,0.629,1728,6


### Women's Bantamweight (20 active fighters)

,Fighter,UFC record,Last fight,Model,Elo,Elo rank
rank,,,,,,
1,Luana Santos,6-1,2026-06-20,0.760,1662,6
2,Ailin Perez,6-1,2026-02-28,0.740,1667,4
3,Jacqueline Cavalcanti,5-1,2026-05-16,0.686,1593,9
4,Joselyne Edwards,9-4,2026-04-25,0.669,1664,5
5,Norma Dumont,9-3,2026-04-25,0.638,1694,2
6,Raquel Pennington,13-6,2024-10-05,0.556,1715,1
7,Karol Rosa,8-5,2026-06-20,0.526,1542,11
8,Nora Cornolle,4-3,2026-09-05,0.523,1551,10
9,Ketlen Vieira,10-5,2026-05-16,0.517,1668,3


## 5. Do the rankings predict the fights?

A ranking claims that a fighter is better than those below. The fights between two ranked fighters test that claim: the better-ranked fighter should win.

The test period is replayed event by event. The day before each event, our rankings are rebuilt from the fights known at that date, with the model trained before the test period. In every fight between two fighters who both hold an official rank in the division (and whom our rankings also rank), each ranking designates a favourite: the better-ranked fighter. The official UFC rankings do the same with the ranks published before the fight, and the betting market with the closing odds.

In [10]:
backtest = backtest_rankings(master, rounds, elo_history, matchups, fitted)
format_summary(backtest)

Rankings against the fights between two ranked fighters (test period):
                   fights  accuracy  disagreements with official rankings  won by this predictor  p-value
predictor                                                                                                
Official rankings     194     0.534                                   NaN                    NaN      NaN
Model                 194     0.722                                  93.0                   65.0    0.000
Elo                   194     0.577                                  73.0                   41.0    0.349
Betting favourite     184     0.717                                  78.0                   57.0    0.000


,Fights,Favourite won,Picks a different fighter than the official rankings,... and is right,p-value
Official rankings,194,53.4%,,,
Model,194,72.2%,93,65,< 0.001
Elo,194,57.7%,73,41,0.349
Betting favourite,184,71.7%,78,57,< 0.001


On the fights between two ranked fighters of the test period, the official rankings designate the winner barely more often than a coin flip (53%). The model ranking does so 72% of the time, level with the betting favourite. On the 93 fights where the model and the official rankings favour different fighters, the model is right 65 times, a split far too uneven to be luck (p < 0.001). Elo, which measures the record, stays close to the official rankings, in its order as in its weakness (58%).

The official rankings are not built to predict: they reward the record and move slowly, whereas the model ranking is built to say who would win today, and the test shows that it does. The test has its limits: with 194 fights each share is known to within about 6 to 7 points, enough to separate the model from the official rankings but not from the market. Details, and a check of the official rankings over 2013-2026: [03_models_and_rankings](notebooks/03_models_and_rankings.ipynb).